# 01 — SISMA data pipeline: connessione DB e path sorgenti Spotify

In [ ]:
import duckdb

source = "../../Downloads/archive (1)/"
db_path = "sisma_core.duckdb"

con = duckdb.connect(db_path)

paths = {
    "audio_features": source + "spotify_clean_audio_features_parquet/track_audio_features.parquet",
    "tracks": source + "spotify_clean_parquet/tracks.parquet",
    "track_artists": source + "spotify_clean_parquet/track_artists.parquet",
    "artists": source + "spotify_clean_parquet/artists.parquet",
    "albums": source + "spotify_clean_parquet/albums.parquet",
}

print("DuckDB pronto:", db_path)
for k, v in paths.items():
    print(k, "->", v)

# 02 — Inizializzazione tabella finale per aggregazione dei chunk

In [ ]:
con.execute("DROP TABLE IF EXISTS tracks_sisma")

con.execute("""
CREATE TABLE tracks_sisma (
    id VARCHAR,
    name VARCHAR,
    popularity BIGINT,
    duration_ms BIGINT,
    explicit BIGINT,
    artists VARCHAR,
    id_artists VARCHAR,
    release_date VARCHAR,
    danceability DOUBLE,
    energy DOUBLE,
    key INTEGER,
    loudness DOUBLE,
    mode INTEGER,
    speechiness DOUBLE,
    acousticness DOUBLE,
    instrumentalness DOUBLE,
    liveness DOUBLE,
    valence DOUBLE,
    tempo DOUBLE,
    time_signature INTEGER
)
""")

# 03 — Setup chunking: range rowid della tabella tracks

In [ ]:
import math
import pandas as pd

chunk_size = 500_000

stats = con.execute(f"""
SELECT MIN(rowid) AS min_rowid, MAX(rowid) AS max_rowid
FROM '{paths["tracks"]}'
""").fetchone()

min_rowid, max_rowid = stats
print("rowid range:", min_rowid, max_rowid)

# 04 — Estrazione, join e append dei chunk nella tabella finale

In [ ]:
for start in range(min_rowid, max_rowid + 1, chunk_size):
    end = start + chunk_size - 1
    print(f"\n=== Chunk {start} - {end} ===")

    con.execute("DROP TABLE IF EXISTS tracks_chunk")
    con.execute("DROP TABLE IF EXISTS track_artists_chunk")
    con.execute("DROP TABLE IF EXISTS artists_chunk")
    con.execute("DROP TABLE IF EXISTS albums_chunk")
    con.execute("DROP TABLE IF EXISTS audio_features_chunk")
    con.execute("DROP TABLE IF EXISTS track_artist_lists_chunk")
    con.execute("DROP TABLE IF EXISTS tracks_sisma_chunk")

    # 1. tracks chunk
    con.execute(f"""
    CREATE TABLE tracks_chunk AS
    SELECT
        rowid AS track_rowid,
        id,
        name,
        album_rowid,
        popularity,
        duration_ms,
        explicit
    FROM '{paths["tracks"]}'
    WHERE rowid BETWEEN {start} AND {end}
      AND id IS NOT NULL
    """)

    n_tracks = con.execute("SELECT COUNT(*) FROM tracks_chunk").fetchone()[0]
    print("tracks_chunk:", n_tracks)

    if n_tracks == 0:
        continue

    # 2. track_artists chunk
    con.execute(f"""
    CREATE TABLE track_artists_chunk AS
    SELECT
        ta.track_rowid,
        ta.artist_rowid
    FROM '{paths["track_artists"]}' ta
    JOIN tracks_chunk t
      ON ta.track_rowid = t.track_rowid
    """)

    # 3. artists chunk
    con.execute(f"""
    CREATE TABLE artists_chunk AS
    SELECT
        a.rowid AS artist_rowid,
        a.id AS artist_id,
        a.name AS artist_name
    FROM '{paths["artists"]}' a
    JOIN (
        SELECT DISTINCT artist_rowid
        FROM track_artists_chunk
    ) z
      ON a.rowid = z.artist_rowid
    WHERE a.id IS NOT NULL
    """)

    # 4. albums chunk
    con.execute(f"""
    CREATE TABLE albums_chunk AS
    SELECT
        al.rowid AS album_rowid,
        al.release_date
    FROM '{paths["albums"]}' al
    JOIN (
        SELECT DISTINCT album_rowid
        FROM tracks_chunk
        WHERE album_rowid IS NOT NULL
    ) z
      ON al.rowid = z.album_rowid
    """)

    # 5. audio_features chunk
    con.execute(f"""
    CREATE TABLE audio_features_chunk AS
    SELECT
        af.track_id AS id,
        CAST(af.duration_ms AS BIGINT) AS duration_ms,
        CAST(af.time_signature AS INTEGER) AS time_signature,
        CAST(af.tempo AS DOUBLE) AS tempo,
        CAST(af.key AS INTEGER) AS key,
        CAST(af.mode AS INTEGER) AS mode,
        CAST(af.danceability AS DOUBLE) AS danceability,
        CAST(af.energy AS DOUBLE) AS energy,
        CAST(af.loudness AS DOUBLE) AS loudness,
        CAST(af.speechiness AS DOUBLE) AS speechiness,
        CAST(af.acousticness AS DOUBLE) AS acousticness,
        CAST(af.instrumentalness AS DOUBLE) AS instrumentalness,
        CAST(af.liveness AS DOUBLE) AS liveness,
        CAST(af.valence AS DOUBLE) AS valence
    FROM '{paths["audio_features"]}' af
    JOIN tracks_chunk t
      ON af.track_id = t.id
    WHERE CAST(af.null_response AS INTEGER) = 0
      AND af.track_id IS NOT NULL
    """)

    # 6. artist lists per chunk
    con.execute("""
    CREATE TABLE track_artist_lists_chunk AS
    SELECT
        ta.track_rowid,
        to_json(list(a.artist_name ORDER BY a.artist_rowid)) AS artists,
        to_json(list(a.artist_id ORDER BY a.artist_rowid)) AS id_artists
    FROM track_artists_chunk ta
    JOIN artists_chunk a
      ON ta.artist_rowid = a.artist_rowid
    GROUP BY ta.track_rowid
    """)

    # 7. build final chunk
    con.execute("""
    CREATE TABLE tracks_sisma_chunk AS
    SELECT
        t.id,
        t.name,
        t.popularity,
        COALESCE(t.duration_ms, af.duration_ms) AS duration_ms,
        t.explicit,
        tal.artists,
        tal.id_artists,
        al.release_date,
        af.danceability,
        af.energy,
        af.key,
        af.loudness,
        af.mode,
        af.speechiness,
        af.acousticness,
        af.instrumentalness,
        af.liveness,
        af.valence,
        af.tempo,
        af.time_signature
    FROM tracks_chunk t
    LEFT JOIN audio_features_chunk af
        ON t.id = af.id
    LEFT JOIN albums_chunk al
        ON t.album_rowid = al.album_rowid
    LEFT JOIN track_artist_lists_chunk tal
        ON t.track_rowid = tal.track_rowid
    """)

    # 8. append
    con.execute("""
    INSERT INTO tracks_sisma
    SELECT * FROM tracks_sisma_chunk
    """)

    total = con.execute("SELECT COUNT(*) FROM tracks_sisma").fetchone()[0]
    print("total rows in tracks_sisma:", total)

# 05 — Esportazione del dataset finale in Parquet

In [ ]:
con.execute("""
COPY tracks_sisma TO 'tracks_sisma.parquet' (FORMAT PARQUET)
""")

# 06 — Creazione del dataset finale pulito (tracks_sisma_final)

In [ ]:
con.execute("""
CREATE TABLE tracks_sisma_final AS
SELECT DISTINCT
    id,
    name,
    popularity,
    duration_ms,
    explicit,
    artists,
    id_artists,
    release_date,
    TRY_CAST(SUBSTR(release_date, 1, 4) AS INTEGER) AS release_year,
    danceability,
    energy,
    key,
    loudness,
    mode,
    speechiness,
    acousticness,
    instrumentalness,
    liveness,
    valence,
    tempo,
    time_signature
FROM tracks_sisma
WHERE id IS NOT NULL
""")

# 07 — Ricostruzione di tracks.csv

In [ ]:
con.execute("""
COPY (
    SELECT
        id,
        name,
        popularity,
        duration_ms,
        explicit,
        artists,
        id_artists,
        release_date,
        danceability,
        energy,
        key,
        loudness,
        mode,
        speechiness,
        acousticness,
        instrumentalness,
        liveness,
        valence,
        tempo,
        time_signature
    FROM tracks_sisma_final
) TO 'tracks.csv' (HEADER, DELIMITER ',')
""")

# 08 — Ricostruzione di artists.csv con aggregazione dei generi

In [ ]:
con.execute("""
COPY (
    WITH genres_agg AS (
        SELECT
            artist_rowid,
            '[' || string_agg(
                concat('''', replace(genre, '''', ''''''), ''''),
                ', ' ORDER BY genre
            ) || ']' AS genres
        FROM '../../Downloads/archive (1)/spotify_clean_parquet/artist_genres.parquet'
        WHERE genre IS NOT NULL
        GROUP BY artist_rowid
    )
    SELECT
        a.id,
        a.followers_total AS followers,
        COALESCE(g.genres, '[]') AS genres,
        a.name,
        a.popularity
    FROM '../../Downloads/archive (1)/spotify_clean_parquet/artists.parquet' a
    LEFT JOIN genres_agg g
      ON a.rowid = g.artist_rowid
    WHERE a.id IS NOT NULL
) TO 'artists.csv' (HEADER, DELIMITER ',')
""")